# F1 PitNextLap — Simple but High-Performance Solution

**Goal:** predict `PitNextLap` for the competition test set and create `submission.csv`.

### What this notebook uses
- No AutoML.
- Only the provided `train.csv`, `test.csv`, and `sample_submission.csv`.
- Strong but easy-to-follow vanilla models:
  1. **LightGBM** — fast gradient boosting.
  2. **CatBoost** — handles categorical columns directly.
- 5-fold stratified cross-validation.
- Final prediction = **rank average** of the two models, which is a good blend for ROC-AUC.

The notebook is intentionally organized as **Load → Understand → Features → Validate → Train → Blend → Submit**.


## 1. Imports and configuration

The main settings are kept in one place.  
If you want a faster run locally, reduce `N_SPLITS` to 3. For the best performance, keep 5.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "PitNextLap"
ID_COL = "id"

# Put train.csv, test.csv and sample_submission.csv in this folder.
DATA_DIR = Path(".")

TRAIN_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\train.csv"
TEST_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\test.csv"
SAMPLE_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\sample_submission.csv"

print("Ready.")


Ready.


## 2. Load the competition data

We use the supplied files directly. No external OOF predictions or external datasets are required.


In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print("Sample:", sample.shape)

display(train.head())


Train: (439140, 16)
Test : (188165, 15)
Sample: (188165, 2)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


## 3. Quick data check

The target is binary.  
Categorical columns are `Driver`, `Compound`, and `Race`; the remaining useful columns are numeric.


In [3]:
print("Target rate:", round(train[TARGET].mean(), 4))
print("\nMissing values:")
display(train.isna().sum().to_frame("missing").T)

print("\nData types:")
display(train.dtypes.to_frame("dtype").T)

print("\nTarget counts:")
display(train[TARGET].value_counts())


Target rate: 0.199

Missing values:


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
missing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



Data types:


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
dtype,int64,str,str,str,int64,int64,int64,int64,float64,int64,float64,float64,float64,float64,float64,float64



Target counts:


PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64

## 4. Feature engineering

We remove only the row identifier and target.

A few simple interaction features are added because pit-stop decisions depend on tyre age, lap position, race progress and stint.

**Important:** we do not use future-lap information or any target-derived feature.


In [4]:
def make_features(df):
    x = df.copy()

    # ID is only an identifier; it should not drive the prediction.
    x = x.drop(columns=[ID_COL], errors="ignore")

    # Simple, interpretable interactions.
    x["TyreLife_sq"] = x["TyreLife"] ** 2
    x["LapNumber_sq"] = x["LapNumber"] ** 2
    x["RaceProgress_sq"] = x["RaceProgress"] ** 2

    x["TyreLife_x_RaceProgress"] = x["TyreLife"] * x["RaceProgress"]
    x["Position_x_RaceProgress"] = x["Position"] * x["RaceProgress"]
    x["Stint_x_TyreLife"] = x["Stint"] * x["TyreLife"]
    x["LapTime_x_TyreLife"] = x["LapTime (s)"] * x["TyreLife"]

    return x

X = make_features(train.drop(columns=[TARGET]))
X_test = make_features(test)

cat_cols = ["Driver", "Compound", "Race"]
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

y = train[TARGET].astype(int)

print("Features:", X.shape[1])
print("Categorical:", cat_cols)
display(X.head())


Features: 21
Categorical: ['Driver', 'Compound', 'Race']


,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),...,Cumulative_Degradation,RaceProgress,Position_Change,TyreLife_sq,LapNumber_sq,RaceProgress_sq,TyreLife_x_RaceProgress,Position_x_RaceProgress,Stint_x_TyreLife,LapTime_x_TyreLife
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,21.019,0.714286,5.0,1521.0,2500,0.510204,27.857143,5.714286,78.0,3061.149
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,-223.207,0.346154,-3.0,49.0,729,0.119822,2.423077,1.384615,14.0,525.665
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,-100.529,0.819444,3.0,484.0,3481,0.671489,18.027778,10.652778,66.0,1560.790
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,-7.324,0.076923,0.0,4.0,4,0.005917,0.153846,0.538462,2.0,188.722
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,-14.139,0.361111,3.0,36.0,676,0.130401,2.166667,0.722222,18.0,647.268


## 5. Cross-validation

We use **StratifiedKFold** so every validation fold has approximately the same positive/negative target ratio.

We keep out-of-fold (OOF) predictions because they give an honest estimate of model quality and allow us to compare the two models before submission.


In [5]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_lgb = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))

oof_cat = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))

print(f"{N_SPLITS}-fold cross-validation started.")


5-fold cross-validation started.


## 6. Model 1 — LightGBM

LightGBM is usually the fastest model for this dataset size and works very well on mixed tabular data.

We use early stopping so the model does not keep adding trees after validation performance stops improving.


In [6]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

    model = LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.10,
        reg_lambda=1.0,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        categorical_feature=cat_cols,
        callbacks=[
            early_stopping(150, verbose=False),
            log_evaluation(0)
        ]
    )

    oof_lgb[valid_idx] = model.predict_proba(X_va)[:, 1]
    pred_lgb += model.predict_proba(X_test)[:, 1] / N_SPLITS

    fold_auc = roc_auc_score(y_va, oof_lgb[valid_idx])
    print(f"Fold {fold}: AUC = {fold_auc:.6f}, best_iter = {model.best_iteration_}")

print(f"\nLightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.6f}")


Fold 1: AUC = 0.943399, best_iter = 801
Fold 2: AUC = 0.941550, best_iter = 645
Fold 3: AUC = 0.941912, best_iter = 657
Fold 4: AUC = 0.941250, best_iter = 580
Fold 5: AUC = 0.942238, best_iter = 714

LightGBM OOF AUC: 0.942065


## 7. Model 2 — CatBoost

CatBoost is a second, structurally different gradient-boosting model.  
It is useful here because it handles `Driver`, `Compound` and `Race` as categorical variables without manual one-hot encoding.

Using a second strong model reduces dependence on one algorithm.


In [7]:
# CatBoost expects categorical values as strings.
X_cat = X.copy()
X_test_cat = X_test.copy()

for c in cat_cols:
    X_cat[c] = X_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_cat, y), 1):
    X_tr, X_va = X_cat.iloc[train_idx], X_cat.iloc[valid_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=2500,
        learning_rate=0.035,
        depth=8,
        l2_leaf_reg=5.0,
        random_seed=RANDOM_STATE + fold,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_tr, y_tr,
        cat_features=cat_cols,
        eval_set=(X_va, y_va),
        use_best_model=True,
        early_stopping_rounds=150,
        verbose=False
    )

    oof_cat[valid_idx] = model.predict_proba(X_va)[:, 1]
    pred_cat += model.predict_proba(X_test_cat)[:, 1] / N_SPLITS

    fold_auc = roc_auc_score(y_va, oof_cat[valid_idx])
    print(f"Fold {fold}: AUC = {fold_auc:.6f}, best_iter = {model.get_best_iteration()}")

print(f"\nCatBoost OOF AUC: {roc_auc_score(y, oof_cat):.6f}")


Fold 1: AUC = 0.949841, best_iter = 2499
Fold 2: AUC = 0.947553, best_iter = 2499
Fold 3: AUC = 0.948607, best_iter = 2493


KeyboardInterrupt: 

## 8. Compare models and create the final blend

ROC-AUC only depends on the ordering of predictions.  
Therefore, instead of averaging two probabilities that may have different scales, we rank-normalize both predictions and average the ranks.

The final submission uses the **rank average**.


In [ ]:
def rank_average(a, b):
    ra = rankdata(a) / len(a)
    rb = rankdata(b) / len(b)
    return 0.5 * ra + 0.5 * rb

auc_lgb = roc_auc_score(y, oof_lgb)
auc_cat = roc_auc_score(y, oof_cat)

raw_blend_oof = 0.5 * oof_lgb + 0.5 * oof_cat
rank_blend_oof = rank_average(oof_lgb, oof_cat)

print(f"LightGBM AUC : {auc_lgb:.6f}")
print(f"CatBoost AUC  : {auc_cat:.6f}")
print(f"Raw blend AUC : {roc_auc_score(y, raw_blend_oof):.6f}")
print(f"Rank blend AUC: {roc_auc_score(y, rank_blend_oof):.6f}")


## 9. Create the final submission

The output has exactly the same `id` order as `sample_submission.csv`.

The file to submit is **`submission.csv`**.


In [ ]:
final_pred = rank_average(pred_lgb, pred_cat)

submission = pd.DataFrame({
    ID_COL: sample[ID_COL],
    TARGET: final_pred
})

# Safety checks before writing.
assert len(submission) == len(sample)
assert submission[ID_COL].equals(sample[ID_COL])
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()

submission.to_csv("submission1111.csv", index=False)

print("Saved: submission.csv")
print("Shape:", submission.shape)
display(submission.head())


## 10. Final result

### Files produced
- **`submission.csv`** → final competition submission.
- The notebook itself documents the complete reproducible pipeline.

### Pipeline in one line
**Raw train/test → simple safe features → 5-fold LightGBM + 5-fold CatBoost → rank average → submission.csv**

No AutoML, no external OOF predictions, and no external datasets are used.
